# Project-Level Fitting

This notebook demonstrates project-level fitting, where multiple datasets are
fitted simultaneously with shared and independent parameters.

**Example System:**
- 6 datasets with identical spectral model (GLP + linear background)
- Different peak shift amplitudes across files (varying pump fluence)
- Shared time dependence: exponential shift of peak position (with IRF)

**Key Concept:**
Parameters are classified by `vary` level in the YAML model:
- `"project"`: shared across all files (fitted once)
- `"file"`: independent per file (fitted separately for each)
- `"static"`: fixed (set from baseline, not fitted)

**Workflow:**
1. Load all datasets into a single Project
2. Fit baseline for each file
3. Load 2D models with project/file/static vary levels
4. Run `project.fit_2d()` to fit all files simultaneously

In [ ]:
import os
import numpy as np
import trspecfit

## 1. Load Data

Create a Project and load all 6 datasets. Each file shares the same energy and
time axes but has a different peak shift amplitude (`expFun A`).

In [ ]:
project = trspecfit.Project(path=os.getcwd())

data_folder = 'data'
energy = np.loadtxt(project.path / data_folder / 'energy.csv')
time = np.loadtxt(project.path / data_folder / 'time.csv')

shift_amplitudes = [5, 2, 1, 0.5, 0.2, 0.1]
files = []

for i, amp in enumerate(shift_amplitudes, start=1):
    file_data = np.loadtxt(project.path / data_folder / f'data_{i}.csv', delimiter=',')
    f = trspecfit.File(
        parent_project=project,
        path=f'data_{i}',
        data=file_data,
        energy=energy,
        time=time,
    )
    files.append(f)
    print(f'File {i}: expFun A = {amp}, shape {file_data.shape}')

project.describe(detail=1)

## 2. Define Fitting Region and Baseline

Set energy and time limits, then define the baseline (ground state) spectrum
for all files at once.

In [ ]:
# Set fitting limits (same for all files)
project.set_fit_limits(
    energy_limits=[5, 18],
    time_limits=[-10, 99],
)

# Define baseline from early time points (before dynamics onset)
project.define_baselines(time_start=0, time_stop=12, time_type='ind')

## 3. Fit Baseline

Load a baseline model and fit it on every file. Since the spectral shape is
identical across files (only the time-dependent shift differs), the baseline
parameters should converge to similar values.

In [ ]:
# Load baseline model onto all files
project.load_models(model_yaml='models_energy.yaml', model_info='base')

# Fit baselines
project.fit_baselines(model_name='base', stages=2)

## 4. Global 2D Fitting

Load the 2D model, add time dependence, and fit all files simultaneously.

The `vary` levels control parameter sharing:

**Energy model** (`models_energy.yaml`, all `"static"`):
- All energy parameters fixed from baseline fit

**Time model** (`models_time.yaml`):
- `"project"`: `tau`, `gaussCONV SD` — shared dynamics across all files
- `"file"`: `expFun A` — independent shift amplitude per file

In [ ]:
# Load 2D model onto all files
project.load_models(model_yaml='models_energy.yaml', model_info='2D')

# Add time dependence to peak position
project.add_time_dependences(
    target_model='2D',
    target_parameter='GLP_01_x0',
    dynamics_yaml='models_time.yaml',
    dynamics_model=['MonoExpPosIRF'],
)

# Fit all files simultaneously
project.fit_2d(model_name='2D', stages=2)